# Exporting BindPredictCNN Models for Triton Inference Server

This notebook guides you through the process of converting a set of 5 cross-validated PyTorch `BindPredictCNN` models into the ONNX format. The exported models are specifically configured to be compatible with NVIDIA Triton Inference Server, enabling high-performance, scalable deployment. 🚀

**The workflow is as follows:**
1.  **Setup**: Install and import necessary libraries.
2.  **Model Definitions**: Define the original and a Triton-specific wrapper for our PyTorch models.
3.  **Load Models**: Load the 5 pre-trained PyTorch model checkpoints.
4.  **Export to ONNX**: Run the export function to convert all 5 models.
5.  **Next Steps**: Review how to structure the Triton model repository for ensemble inference.

---

## 1. Setup & Prerequisites

First, let's install the required Python libraries. You'll need `torch` for model handling and `onnx` / `onnxruntime` for the conversion.

In [2]:
import torch
import torch.nn as nn
import os
from pathlib import Path
import numpy as np

import sys
sys.path.append('..')
from prott5_batch_predictor import BindPredict

print(f"PyTorch version: {torch.__version__}")

/Users/t03i/Workspace/RostLabScience/prott5_predictors/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch version: 2.8.0


---

## 2. Model Architecture Definitions

For this notebook to be self-contained, we need to define the model architecture here. We'll define:
1. `TritonBindPredictCNN`: A wrapper class that includes the final permutation step inside the `forward` pass. This makes the exported ONNX graph simpler and more efficient for Triton.


In [3]:
class TritonBindPredictCNN(nn.Module):
    """A wrapper to make the BindPredictCNN model Triton-friendly."""
    def __init__(self, original_model):
        super().__init__()
        # We only need the convolutional part from the original model.
        # The original model's forward pass returns a squeezed tensor,
        # but for export, we need the raw convolutional output.
        self.conv1 = original_model.conv1

    def forward(self, x):
        """
        Takes an input of shape (batch_size, 1024, sequence_length) 
        and returns logits of shape (batch_size, sequence_length, 3).
        """
        logits = self.conv1(x)  # Output shape: (batch_size, 3, sequence_length)
        
        # Permute to (batch_size, sequence_length, 3) for easier handling on the client-side.
        return torch.permute(logits, (0, 2, 1))

---

## 3. Define the ONNX Export Function

This function will iterate through our 5 loaded PyTorch models, wrap each one, and export it with the correct dynamic axes for Triton.

In [4]:
def export_bindpredict_to_triton_onnx(pytorch_models, onnx_dir_path):
    """
    Exports a list of PyTorch models to Triton-compatible ONNX format.
    
    Args:
        pytorch_models (list): A list of loaded PyTorch BindPredictCNN models.
        onnx_dir_path (str or Path): The directory to save the exported .onnx files.
    """
    # Create the output directory if it doesn't exist
    if not os.path.exists(onnx_dir_path):
        os.makedirs(onnx_dir_path)
        print(f"✅ Created directory: {onnx_dir_path}")

    for i, model in enumerate(pytorch_models):
        # Wrap the original model for Triton-friendly export
        model_for_export = TritonBindPredictCNN(model)
        model_for_export.eval() # Set to evaluation mode

        # Create a dummy input tensor to trace the model's execution graph.
        # The dimensions represent (batch_size, embedding_dim, sequence_length).
        batch_size = 1       # For tracing, batch can be 1
        embedding_dim = 1024 # This dimension is fixed
        sequence_length = 100 # An example sequence length
        dummy_input = torch.randn(batch_size, embedding_dim, sequence_length, requires_grad=False)

        # Define the full path for the output ONNX file
        onnx_file_path = os.path.join(onnx_dir_path, f'model_{i}.onnx')

        print(f"\nExporting model {i} to {onnx_file_path}...")

        # Export the model to ONNX format
        torch.onnx.export(
            model_for_export,               # The model to export
            dummy_input,                    # A dummy input for tracing
            onnx_file_path,                 # Where to save the model
            export_params=True,             # Store the trained weights within the file
            opset_version=12,               # The ONNX version to use
            do_constant_folding=True,       # Apply optimizations
            input_names=['input_embedding'],  # The name for the input tensor
            output_names=['output_logits'], # The name for the output tensor
            dynamic_axes={                  # Define which dimensions can vary in size
                'input_embedding': {0: 'batch_size', 2: 'sequence_length'},
                'output_logits': {0: 'batch_size', 1: 'sequence_length'}
            }
        )
        print(f"✅ Successfully exported model {i}.")

---

## 4. Load Models and Run Export

Now we'll define our paths, load the 5 PyTorch models using the `BindPredict` class, and call our export function.

In [ ]:
# --- 1. Define Paths ---
# Adjust these paths if your directory structure is different.
root_dir = Path.cwd()
model_checkpoints_dir = root_dir / "../checkpoints"
onnx_export_dir = model_checkpoints_dir / "bindpredict_onnx_triton"

print(f"Looking for PyTorch checkpoints in: {model_checkpoints_dir}")
print(f"Exported ONNX models will be saved to: {onnx_export_dir}")

# --- 2. Load PyTorch Models ---
print("\nLoading the 5 pre-trained PyTorch models...")
# This assumes the BindPredict class downloads or finds the models in 'model_checkpoints_dir'
bind_models = BindPredict(model_dir=model_checkpoints_dir).model
print(f"✅ Loaded {len(bind_models)} models successfully.")

# --- 3. Run the Export ---
export_bindpredict_to_triton_onnx(pytorch_models=bind_models, onnx_dir_path=onnx_export_dir)

Looking for PyTorch checkpoints in: /Users/t03i/Workspace/RostLabScience/prott5_predictors/onnx_export_notebooks/checkpoints
Exported ONNX models will be saved to: /Users/t03i/Workspace/RostLabScience/prott5_predictors/onnx_export_notebooks/checkpoints/bindpredict_onnx_triton

Loading the 5 pre-trained PyTorch models...


FileNotFoundError: [Errno 2] No such file or directory: '/Users/t03i/Workspace/RostLabScience/prott5_predictors/onnx_export_notebooks/checkpoints/bindpredict'

---